# 20. Proof, Not Promises

## 1. The pain

Every fidelity estimate you have ever been shown was a bare number: no error bar, no statement of where it was validated, and no record of whether the last one turned out to be right. This notebook closes that loop three ways: the estimate ships with its measured accuracy band, you can check it against a mirror-circuit run, and every compile leaves a receipt that future compiles are compared against.

In [1]:
import os
import tempfile
import warnings

# Keep the demo self-contained: verify records and receipts go to a temp dir,
# not ~/.qb_compiler. Delete this line to keep a real local history.
os.environ["QBC_DATA_DIR"] = tempfile.mkdtemp(prefix="qbc_demo_")

# Silence an unrelated urllib3 version warning from this environment.
warnings.filterwarnings("ignore", message=r"urllib3")

import qb_compiler

print("qb-compiler", qb_compiler.__version__)

qb-compiler 0.5.1.post1.dev1+g42eddfbf9.d20260430


## 2. The band

Start with a plain preflight on a Bell pair targeting ibm_fez.

In [2]:
from qiskit import QuantumCircuit

from qb_compiler import check_viability

bell = QuantumCircuit(2, name="Bell")
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

pre = check_viability(bell, backend="ibm_fez")
print(pre)

Circuit: Bell
Backend: ibm_fez
Estimated fidelity: 0.9430 (+-0.05 typical, model tends to overestimate; validated on n=6 hw circuits (Fez, GHZ family))
Noise floor: 0.2500
Signal/noise ratio: 3.8x
Status: VIABLE
Reason: Estimated fidelity 0.943 is well above noise floor (0.2500). Circuit should produce meaningful results.
Calibration snapshot age: 90 days
Error budget:
  - readout: 0.0559 (98% of loss)
  - two_qubit_gates: 0.0012 (2% of loss)
Cost (4096 shots): $0.6554
Suggestions:
  - Circuit looks good: proceed with execution.
  - Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.


Look at the fidelity line: the estimate never appears alone. The +-0.05 typical error comes from n=6 predicted-vs-measured hardware pairs (IBM Fez, GHZ family, March 2026), and on that set the model tends to be optimistic. Small validation set, one backend, one circuit family: indicative, not a guarantee. Stating that is the whole point. A number with a measured band and a known bias is something you can act on; a bare number is a promise.

## 3. Verify mode

A band is only honest if you can test it. `build_mirror` strips the final measurements, appends the inverse of the circuit, and measures everything: under noiseless execution the mirror returns all zeros with probability 1, so the all-zeros rate is a success proxy related to (not equal to) the squared fidelity of the original circuit.

In [3]:
from qb_compiler import build_mirror

mirror = build_mirror(bell)
print(mirror.draw(output="text"))

        ┌───┐          ┌───┐ ░ ┌─┐   
   q_0: ┤ H ├──■────■──┤ H ├─░─┤M├───
        └───┘┌─┴─┐┌─┴─┐└───┘ ░ └╥┘┌─┐
   q_1: ─────┤ X ├┤ X ├──────░──╫─┤M├
             └───┘└───┘      ░  ║ └╥┘
meas: 2/════════════════════════╩══╩═
                                0  1 


In [4]:
from qb_compiler import verify_viability

result = verify_viability(bell, "aer", backend="ibm_fez", shots=1024)
print(result)

Verify (ibm_fez, 2q, 1024 shots): predicted^2 0.8892 vs measured 1.0000 (discrepancy +0.1108)


Read the comparison carefully. The prediction models ibm_fez hardware, but the runner here is a noiseless Aer simulator, so measured 1.0000 against predicted^2 0.8892 is exactly the right answer: the +0.11 discrepancy is the noise the model expects Fez to add and Aer does not have. Point the same call at real hardware (any SamplerV2-style runner works as the second argument) and the discrepancy becomes a genuine accuracy check on the model. One caveat stays either way: the mirror proxy can be optimistic when coherent errors echo out between the circuit and its inverse, and the docstring says so.

## 4. The accuracy record

Every verify call appends a predicted-vs-measured pair to a local log. Run two more circuits, then summarise.

In [5]:
def make_ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, name=f"GHZ-{n}")
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc


print(verify_viability(make_ghz(4), "aer", backend="ibm_fez", shots=1024))
print(verify_viability(make_ghz(6), "aer", backend="ibm_fez", shots=1024))

Verify (ibm_fez, 4q, 1024 shots): predicted^2 0.7693 vs measured 1.0000 (discrepancy +0.2307)


Verify (ibm_fez, 6q, 1024 shots): predicted^2 0.7065 vs measured 1.0000 (discrepancy +0.2935)


In [6]:
from qb_compiler.verify import accuracy_summary

summary = accuracy_summary()
for k, v in summary.items():
    print(f"{k}: {v}")

n: 3
median_abs_discrepancy: 0.23068857318400005
mean_signed_discrepancy: 0.2116543046356667
per_backend_counts: {'ibm_fez': 3}


Three records, median absolute discrepancy about 0.23, all positive because every run so far was noiseless: on this simulator the summary measures the hardware noise the model prices in, not the model's error. Feed it hardware runs and the same summary becomes your own accuracy ledger for the fidelity model, built from your circuits on your backends. The record grows with use and stays on your machine: a plain JSONL file under `QBC_DATA_DIR` (default `~/.qb_compiler`), nothing transmitted anywhere.

## 5. Receipts

The third loop: every compile can leave a passport. `make_receipt` captures what was compiled, for which backend, what was predicted (band attached), where the error went, how old the calibration was, and which tool versions produced all of it.

In [7]:
import json

from qb_compiler import make_receipt, record_receipt

ghz6 = make_ghz(6)
pre6 = check_viability(ghz6, backend="ibm_fez")

receipt = make_receipt(ghz6, pre6, backend="ibm_fez")
print(receipt)
print()
print(json.dumps(receipt.to_json_dict(), indent=2))

Receipt[74ad9b558f743804] GHZ-6 on ibm_fez @ 2026-06-12T11:40:19.015624+00:00: 6q depth 7 -> 5 2q gates, depth 12, predicted fidelity 0.8405 (+-0.05 typical) (qiskit 1.4.5, qb-compiler 0.5.1.post1.dev1+g42eddfbf9.d20260430)

{
  "timestamp": "2026-06-12T11:40:19.015624+00:00",
  "backend": "ibm_fez",
  "structural_hash": "74ad9b558f743804",
  "circuit_name": "GHZ-6",
  "n_qubits": 6,
  "depth_in": 7,
  "post_2q_count": 5,
  "post_depth": 12,
  "predicted_fidelity": 0.840534,
  "fidelity_typical_abs_error": 0.05,
  "error_budget": {
    "two_qubit_gates": 0.009892,
    "readout": 0.151068
  },
  "calibration_age_days": 90.1,
  "qiskit_version": "1.4.5",
  "qb_compiler_version": "0.5.1.post1.dev1+g42eddfbf9.d20260430",
  "seed": null,
  "layout": null,
  "notes": []
}


The structural hash fingerprints the workload, deliberately coarsely: parameter values are excluded, so two iterations of the same ansatz hash identically and can be compared across sessions. Check the receipt against history, then record it.

In [8]:
from qb_compiler.receipts import regression_check

report = regression_check(receipt)
print(report.status)
print(report.message)

record_receipt(receipt)

NO_BASELINE
No prior receipt for workload 74ad9b558f743804 on ibm_fez; recorded as the new baseline.


## 6. Regression watch

`NO_BASELINE`: first sighting of this workload, so it becomes the baseline. Now simulate next week. We fabricate a second receipt with `dataclasses.replace`, dropping the predicted fidelity by 0.15: this stands in for what a drifted calibration snapshot would produce for the same circuit, without waiting a week for real drift.

In [9]:
import dataclasses

degraded = dataclasses.replace(
    receipt,
    predicted_fidelity=receipt.predicted_fidelity - 0.15,
    notes=["fabricated for the demo: simulates a drifted calibration snapshot"],
)

report = regression_check(degraded)
print(report.status)
print(report.message)

REGRESSION
Predicted fidelity 0.6905 vs 0.8405 at baseline (2026-06-12T11:40:19.015624+00:00): delta -0.1500 dropped beyond the combined typical-error band (+-0.1000); two-qubit gate count changed by +0.


`REGRESSION`: the 0.15 drop exceeds the combined +-0.10 band of the two estimates, so it is flagged as a real change rather than estimate noise. A drop inside the band would read `STABLE`: the bands from section 2 are what keep this check from crying wolf. And note what did not happen: nothing was blocked, no threshold was applied, no job was cancelled. Your compilation has a memory; you decide what to do with it.

## 7. Close

Bands, verify mode, receipts, and regression watch are free signals in `pip install qb-compiler` and stay on your machine. The layer that turns them into signed certificates and execution warranties is the QubitBoost SDK.